# Tier 2 N=5000 — Re-rodar FT-CUR + SAINT com val_loss

Após a ablação H2 confirmar estatisticamente que `val_loss > val_acc`
para esses modelos (Wilcoxon $p < 10^{-9}$), este experimento revalida
FT-CUR e SAINT em todos os 6 datasets do Tier 2 com o protocolo corrigido.

**Modelos**: FT-CUR (Nyströmformer) + SAINT  
**Datasets**: ADULT, CREDIT, BANK, TELCO, SHOPPERS, HIGGS50K  
**Métrica de early stopping**: `val_loss`  
**Sementes**: 30  
**Total**: 12 tunings + 360 runs

**Tempo esperado:**
- T4 GPU: ~5-7h (tuning ~3-4h + experimentos ~2-3h)
- A100: ~2-3h

**Antes de rodar:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ── Célula 1: GPU check ─────────────────────────────────────────────────────
!nvidia-smi -L
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  VRAM: {p.total_memory/1e9:.1f} GB')
else:
    print('⚠️ GPU não disponível — Runtime → Change runtime type → GPU')

In [ ]:
# ── Célula 2: Clonar do GitHub ──────────────────────────────────────────────
import os
PROJECT_DIR = '/content/sparse-lssvm-transformers-study'
GIT_URL = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'

if os.path.exists(PROJECT_DIR):
    !cd {PROJECT_DIR} && git pull --rebase
else:
    !git clone {GIT_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
!git log --oneline -3

In [ ]:
# ── Célula 3: Dependências ──────────────────────────────────────────────────
!pip install -q optuna entmax xgboost

import numpy, scipy, sklearn, torch
print(f'numpy {numpy.__version__} | torch {torch.__version__}')

In [ ]:
# ── Célula 4: Baixar todos os 6 datasets do Tier 2 ─────────────────────────
!python scripts/download_data.py 2>&1 | tail -10

In [ ]:
# ── Célula 5: Montar Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/dissertacao_tier2'
import os
os.makedirs(DRIVE_PATH, exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/tuning', exist_ok=True)
print(f'Drive: {DRIVE_PATH}')

In [ ]:
# ── Célula 6: Restaurar progresso anterior (resume) ─────────────────────────
import shutil
from pathlib import Path

drive_results = Path(DRIVE_PATH)
local_results = Path('results')
local_results.mkdir(exist_ok=True)
(local_results / 'tuning').mkdir(exist_ok=True)

# Resultados de experimentos (val_loss)
src = drive_results / 'tier2_n5000_ftcur_saint_valloss.json'
if src.exists():
    shutil.copy(src, 'results/tier2_n5000_ftcur_saint_valloss.json')
    print('✓ Restaurado: tier2_n5000_ftcur_saint_valloss.json')
else:
    print('• Começando do zero (experimentos)')

# Params tunados com val_loss
src_p = drive_results / 'tuning' / 'best_params_ftcur_saint_valloss.json'
if src_p.exists():
    shutil.copy(src_p, 'results/tuning/best_params_ftcur_saint_valloss.json')
    import json
    n = len(json.load(open('results/tuning/best_params_ftcur_saint_valloss.json')))
    print(f'✓ Restaurado: {n} combos tunados com val_loss')
else:
    print('• Tuning vai começar do zero')

In [ ]:
# ── Célula 7: Sync para Drive a cada 5 min (em background) ──────────────────
%%writefile /content/sync_to_drive.sh
#!/bin/bash
while true; do
    sleep 300
    cp -u /content/sparse-lssvm-transformers-study/results/tier2_n5000_ftcur_saint_valloss.json \
          "$1/tier2_n5000_ftcur_saint_valloss.json" 2>/dev/null
    mkdir -p "$1/tuning"
    cp -u /content/sparse-lssvm-transformers-study/results/tuning/best_params_ftcur_saint_valloss.json \
          "$1/tuning/best_params_ftcur_saint_valloss.json" 2>/dev/null
done

In [ ]:
import subprocess
sync_proc = subprocess.Popen(['bash', '/content/sync_to_drive.sh', DRIVE_PATH])
print(f'Sync rodando (PID {sync_proc.pid}) — salva a cada 5 min')

In [ ]:
# ── Célula 8: Rodar tuning + experimentos ───────────────────────────────────
# Tuning: 2 modelos × 6 datasets × 15 trials × 3-fold CV = 12 combos
# Experimentos: 2 × 6 × 30 = 360 runs
#
# Em T4: ~5-7h total (tuning ~3-4h + experimentos ~2-3h)
# Em A100: ~2-3h

!python scripts/run_ftcur_saint_rerun.py \
    --early-stop-metric val_loss \
    --datasets ADULT CREDIT BANK TELCO SHOPPERS HIGGS50K \
    --output-file results/tier2_n5000_ftcur_saint_valloss.json \
    --params-file results/tuning/best_params_ftcur_saint_valloss.json \
    --seeds 30 \
    --trials 15 \
    --folds 3

In [ ]:
# ── Célula 9: Salvar resultado final no Drive ───────────────────────────────
import shutil, signal
try:
    sync_proc.send_signal(signal.SIGTERM)
except Exception:
    pass

from pathlib import Path
import os
drive_dest = Path(DRIVE_PATH)
(drive_dest / 'tuning').mkdir(exist_ok=True)

for fname in ['tier2_n5000_ftcur_saint_valloss.json',
              'tuning/best_params_ftcur_saint_valloss.json']:
    src = Path('results') / fname
    dst = drive_dest / fname
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dst)
        print(f'✓ Salvo: {dst.name} ({src.stat().st_size / 1024:.1f} KB)')

print(f'\nConteúdo do Drive ({DRIVE_PATH}):')
!ls -lh '{DRIVE_PATH}'

In [ ]:
# ── Célula 10: Análise final — comparação val_acc vs val_loss em Tier 2 ────
import json, numpy as np
from collections import defaultdict

# Resultados novos (val_loss)
data_new = json.load(open('results/tier2_n5000_ftcur_saint_valloss.json'))
scores_new = defaultdict(list)
for r in data_new:
    if r.get('status') != 'ok': continue
    scores_new[(r['model_variant'], r['dataset'])].append(r.get('f1_macro', float('nan')))

# Resultados antigos (val_acc), do tier2_n5000_cpu.json se disponível
import os
old_path = 'results/tier2_n5000_cpu.json'
scores_old = defaultdict(list)
if os.path.exists(old_path):
    for r in json.load(open(old_path)):
        if r.get('status') != 'ok': continue
        m = r.get('model_variant') or r.get('model')
        if m in {'FTTransformerCURColnorm', 'SAINTColnorm'}:
            scores_old[(m, r['dataset'])].append(r.get('f1_macro', float('nan')))

DATASETS = ['ADULT', 'CREDIT', 'BANK', 'TELCO', 'SHOPPERS', 'HIGGS50K']
MODELS = ['FTTransformerCURColnorm', 'SAINTColnorm']

print('=' * 95)
print('Tier 2 N=5000 — FT-CUR e SAINT: val_acc (original) vs val_loss (corrigido)')
print('=' * 95)
print(f'\n{"Modelo":<26} {"Dataset":<10} {"val_acc":>10} {"val_loss":>10} {"Δ":>10}')
print('─' * 75)
for m in MODELS:
    for ds in DATASETS:
        old = np.mean(scores_old.get((m, ds), [float('nan')]))
        new = np.mean(scores_new.get((m, ds), [float('nan')]))
        delta = new - old if not (np.isnan(old) or np.isnan(new)) else float('nan')
        old_s = f'{old:>10.4f}' if not np.isnan(old) else f'{"—":>10}'
        new_s = f'{new:>10.4f}' if not np.isnan(new) else f'{"—":>10}'
        d_s = f'{delta:>+10.4f}' if not np.isnan(delta) else f'{"—":>10}'
        print(f'{m:<26} {ds:<10} {old_s} {new_s} {d_s}')

# Resumo agregado
print('\n=== Média sobre os 6 datasets ===')
for m in MODELS:
    all_old = [v for ds in DATASETS for v in scores_old.get((m, ds), [])]
    all_new = [v for ds in DATASETS for v in scores_new.get((m, ds), [])]
    o = np.mean(all_old) if all_old else float('nan')
    n = np.mean(all_new) if all_new else float('nan')
    print(f'  {m:<26}  val_acc={o:.4f}  val_loss={n:.4f}  Δ={n-o:+.4f}')